*© Sabrine Bendimerad · [sbendimerad.github.io](https://sbendimerad.github.io) · Licence [CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/): you may share and adapt this material with credit, for non-commercial use, under the same licence. Datasets keep their own licences.*

# Topic modeling on newsgroups 📰

In this exercise, you run LSA on 2,400 messages to extract topics without any label. Then you compare the result with the newsgroup where each message was posted. We never use the newsgroup to train a model: only to check whether the topics found by LSA look like the categories a human would have chosen.

## The dataset 🗂️
`data/newsgroups.csv` contains 2,400 messages of the *20 Newsgroups* dataset (Usenet discussion groups of the early 1990s, collected by Ken Lang), 400 from each of 6 groups: `comp.graphics`, `rec.autos`, `rec.sport.hockey`, `sci.med`, `sci.space` and `talk.politics.mideast`. Columns: `text` (the message, with headers, signatures and quoted replies removed) and `newsgroup`.

## Loading the data
1. Import the libraries.

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import spacy
from sklearn.cluster import KMeans
from sklearn.decomposition import NMF, TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import Normalizer

import plotly.io as pio
import plotly.graph_objects as go
# colour palette of the course
pio.templates["course"] = go.layout.Template(
    layout_colorway=["#3B47C4", "#E4572E", "#2E7D4F", "#F2C14E", "#8E7CC3", "#4BA3C3", "#111111", "#999999"]
)
pio.templates.default = "plotly_white+course"

2. Load the dataset. How many messages are there in each newsgroup?

In [ ]:
news = pd.read_csv("data/newsgroups.csv")
news.head()

,text,newsgroup
0,What would all of you out there in net land th...,sci.space
1,I aparantly mistyped the address for the ftp s...,comp.graphics
2,Also remember that every dollar spent keeping ...,sci.space
3,This is ok in my opinion as long as the stuff ...,sci.space
4,I was skimming through a few gophers and bumpe...,comp.graphics


## Preparing the texts
3. Create a column `clean`: the text in lowercase, with only the letters (replace everything else by a space).

💡 Hint: `news["text"].str.lower().str.replace(r"[^a-z]+", " ", regex=True)`.

4. Load spaCy's `en_core_web_sm` model, without the components you don't need (`disable=["parser", "ner"]`). Create a column `lemmas` that contains, for each message, the lemmas of its tokens that are not stop words and have at least 3 letters, joined by spaces.

💡 Hint: `nlp.pipe(news["clean"], batch_size=200)` processes the texts in batches, much faster than calling `nlp` on each text. A token has the attributes `.lemma_` and `.is_stop`, and `len(token)` gives its number of characters.

5. Build the TF-IDF matrix `X` with `TfidfVectorizer(min_df=5, max_df=0.5)`. What is its shape, and what share of its values are zeros?

## LSA
6. Run a `TruncatedSVD` with 10 topics (`random_state=0`) and store the coordinates of the messages in `doc_topics`. What share of the variance do the 10 topics keep? What do you notice?

7. For each topic, display the 8 words with the largest weights and the 5 words with the smallest (most negative) weights. Give a name to the topics you can.

💡 Hint: `svd.components_[j]` holds the weights of the words in topic `j`; `np.argsort(weights)[::-1][:8]` gives the positions of the 8 largest.

## Is each message about its strongest topic?
8. Assign each message to the topic where its coordinate is the largest (`argmax`), in a column `topic`. How many messages does each topic get? What do you notice?

9. Cross the newsgroups with the topics (`pd.crosstab`) and draw the table as a heatmap. Compute its **purity**: for each topic, count the messages of its most common newsgroup, add these counts up and divide by the number of messages (1 = each topic contains a single newsgroup). Do the topics match the newsgroups?

💡 Hint: `table = pd.crosstab(news["newsgroup"], news["topic"])`, `px.imshow(table, text_auto=True)`, and the purity is `table.max().sum() / table.values.sum()`.

10. Why does topic 1 win so often? Check: what share of the messages have a positive coordinate on topic 1, and what are the average coordinates of each newsgroup on the 10 topics?

## Clustering in the topic space
A topic is a direction, not a group. To make groups, let's run a clustering on the topic coordinates.

11. Normalize the rows of `doc_topics` with `Normalizer()` (each message becomes a vector of length 1), then run `KMeans` with 6 clusters (`n_init=10`, `random_state=0`). Cross the clusters with the newsgroups, draw the heatmap and compute the purity. Compare with the purity of the same k-means without the `Normalizer`.

💡 Hint: after `Normalizer`, the distance between two messages only depends on the angle between them (their cosine similarity): k-means compares the *mix of topics* of the messages, not how long or how "ordinary" they are.

12. Which newsgroups does k-means find well? Which ones does it mix up, and why could that be?

## Bonus: NMF
13. Run an `NMF` with 6 topics (`init="nndsvda"`, `max_iter=500`, `random_state=0`) on `X`. Display the 8 words with the largest weights of each topic, assign each message to its strongest NMF topic, and compute the purity. Compare with LSA.

## Conclusion
14. What do you conclude about the topics found by LSA and the newsgroups?